# C4 — the per-token model where it matters, on a real T4

**Why.** Two results of 2026-10-05 could not be checked from data on disk.
- **T1** found the simulator's per-token latency within 7% of a live engine on three runs — and
  1.5–1.7× too slow on the two at the knee of the load curve. There, the model's decode cost is a
  straight line drawn several times past the largest batch any calibration ever measured.
- **T2** sized the fleet under a target with both halves (first token *and* time per token) — in a
  regime, below 100 ms per token, where no traffic had ever been measured.

| Part | What it measures |
|---|---|
| **1** | `kvsim calibrate`, with its repeats (seed 0) |
| **2** | decode steps of batches that **share a cached prefix** — 7 batch sizes × 2, up to a summed context 9× the largest ever fitted — and 2 unique-prompt controls |
| **2b** | the long gap every decoding sequence sees when one more request arrives and is prefilled — 3 cells × 2 |
| **3** | 3 held-out workloads around the per-token limit, every request's token times kept, the engine's own histograms read before and after |
| **4** | `kvsim calibrate` again (seeds 1, 2), after the traffic |

**Registered before the run** (`configs/c4_per_token.yaml`, `docs/09-hypotheses.md`):

| | Rule | Predicted |
|---|---|---|
| **C4a** | `kvsim calibrate` exits 0, its cache check is exact, it reads the capacity, and its profile carries the fit of each of its 3 repeats | holds |
| **D1a** | a batch that shares a prefix costs the same per step as one with unique prompts of the same length (within 1.1×) | holds |
| **D1b** | the model's decode step is within 1.25× of the measured one for every shared batch past the largest ever fitted | holds |
| **D2** | when one more request arrives, the extra gap every decoding sequence sees is within 1.5× of what the model charges | holds |
| **T3a** | the model's p95 time per output token is within 1.25× of the measured one, on every workload | holds |
| **T3b** | the model's verdict on the 100 ms limit equals the engine's own, on `long_below` and `long_above` | holds |
| **E** | the spread `kvsim replay` prints from one run's repeats covers another run's count in ≥ 90% of cases (decided afterwards, on a CPU) | holds |
| **F1 · F2** | the 64-prompt question on the fresh calibrations, under both targets (decided afterwards, on a CPU) | holds · holds |

The command comes from the package's wheel, **embedded in this notebook** (sha256
`36725240737e…`, built from commit `c5b5d74`). This notebook decides nothing: it measures,
and `scripts/c4_analyze.py` decides.


**How to run.** `python scripts/kaggle_run.py notebooks/c4_per_token_t4.ipynb --slug
kvsim-c4-per-token-t4 --out <folder>` — or by hand: Kaggle → *Settings* → Accelerator **GPU T4 ×2**
(one T4 is used), Internet **on** → *Run All*. About 67 minutes. Results are saved after
every part, so a stop with ⛔ still leaves **`c4_results.zip`** with the parts before it. Run it
once: a second run is a second experiment.


## 0 — Configuration


In [ ]:
import json

# ── Generated from configs/c4_per_token.yaml. Do not edit here: edit the config and re-run
#    scripts/c4_prepare.py, or the registration stops describing this notebook.
MODEL = "Qwen/Qwen3-4B"
VLLM_PINNED = "0.30.0"
BLOCK_SIZE = 16
NUM_GPU_BLOCKS = 1537          # --num-gpu-blocks-override; one is vLLM's null block
MAX_MODEL_LEN = 2048
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 8192
N_REPLICAS = 1
MEASURE = 0                 # everything runs against this replica
CAL = json.loads(r"""{"seeds": [0, 1, 2], "check_sessions": 16, "check_turns": 3, "token_range": [1000, 30000], "max_error": 0.05}""")     # the command's settings; its timing cells are its defaults
DESIGN = json.loads(r"""{"decode": {"output_tokens": 64, "skip_tokens": 8}, "decode_shared": {"prefix_tokens": 1360, "suffix_tokens": 48, "output_tokens": 64, "skip_tokens": 8, "reps": 2, "batches": [8, 12, 16, 32, 64, 96, 128], "controls": [[1408, 8], [1408, 12]]}, "stall": {"output_tokens": 96, "reps": 2, "cells": [[8, 128, 2.0], [32, 128, 5.0], [64, 128, 9.5]]}}""")
TPOT_LIMIT_S = 0.1      # the per-token half of the target, in seconds
WHEEL_SHA256 = "36725240737eda86aa02b9273d80eafe06727e36d39ec037d02650924831b8f0"

# ── Box settings. Safe to change: nothing predicted depends on them.
GPU_MEM_UTIL = 0.85
PORTS = [8000]                # replica i listens on PORTS[i] and runs on GPU i
STARTUP_TIMEOUT_S = 1800      # the first start downloads ~8 GB of weights and compiles
OUT_DIR = "c4_results"


## 1 — Environment


In [ ]:
import hashlib
import json
import os
import pathlib
import platform
import random
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

pathlib.Path(OUT_DIR).mkdir(exist_ok=True)


class Stop(Exception):
    """Halts Run-All with a reason. Download phase6_results.zip anyway."""


def sh(cmd, timeout=60):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True,
                              timeout=timeout, check=False).stdout.strip()
    except Exception as e:  # an unfamiliar box: any binary may be missing
        return f"<failed: {e}>"


ENV = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "cpu_count": os.cpu_count(),
    "gpus": sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version "
               "--format=csv,noheader"),
    "kaggle": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
}
for k, v in ENV.items():
    print(f"{k:10s} {v}")
_n_gpus = len([ln for ln in ENV["gpus"].splitlines() if ln.strip() and "failed" not in ln])
if _n_gpus < N_REPLICAS:
    raise Stop(f"⛔ {_n_gpus} GPU(s) visible, {N_REPLICAS} needed: one server per GPU is the "
               "whole point. Kaggle: Settings → Accelerator → GPU T4 ×2.")
if "T4" not in ENV["gpus"]:
    print("\n⚠️  Not a Tesla T4. Hit rates stay valid (cache counts are hardware-independent), "
          "but every timing must be labelled with the GPU above, not as a T4 figure.")


## 2 — Install vLLM, pinned — and kv-routing-sim, from the wheel embedded here


In [ ]:
# PINNED: the predictions were made for this version, and Phase 0 v2 passed every check on it.
!pip install -q vllm==0.30.0


## 2b — Pre-flight: can this box's software start the engine at all?


In [ ]:
# ── Pre-flight (added 2026-10-05, docs/09). Can this box's software start vLLM at all? Answered in
#    about a minute, before the 8 GB download — and repaired where the repair is known.
#
#    Why it exists: `pip install vllm` brings the torch build vLLM pins, but KEEPS any package whose
#    version number already satisfies a pin. Kaggle's image ships torchaudio 2.11.0 built for another
#    CUDA than that torch — the very version vLLM pins — so pip keeps it, and then nothing that
#    imports transformers can start: vLLM stops with "PyTorch and TorchAudio were compiled with
#    different CUDA versions". The repair puts back the build from PyPI, which is what pip installs
#    on a clean box. vLLM itself, its version and its flags are not touched.
import importlib
import importlib.metadata
import zipfile

PREFLIGHT = {"repairs": []}
_FAMILY = ("torchaudio", "torchvision", "torchcodec")  # none is needed by a text model; any can break

# Run in a FRESH interpreter: is the module installed, and does importing it work?
_PROBE = """
import importlib, importlib.metadata as md, importlib.util, json, sys
mod = sys.argv[1]
if importlib.util.find_spec(mod) is None:
    print("PROBE " + json.dumps({"installed": False}))
    sys.exit(0)
try:
    ver = md.version(mod)
except Exception:
    ver = None
try:
    importlib.import_module(mod)
    out = {"installed": True, "version": ver, "ok": True}
except BaseException as e:
    out = {"installed": True, "version": ver, "ok": False,
           "error": (type(e).__name__ + ": " + str(e))[:500]}
print("PROBE " + json.dumps(out))
"""
_TORCH = """
import json, torch
ok = torch.cuda.is_available()
print("PROBE " + json.dumps({"version": torch.__version__, "cuda_build": torch.version.cuda,
                             "cuda_available": ok,
                             "device": torch.cuda.get_device_name(0) if ok else None}))
"""


def _run(cmd, timeout=1200):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=False)
        return p.returncode, p.stdout + p.stderr
    except Exception as e:  # a missing binary or a timeout is an answer too
        return -1, repr(e)


def _answer(code, *args):
    """The PROBE line a fresh interpreter printed, or why there is none."""
    rc, out = _run([sys.executable, "-c", code, *args], 600)
    line = next((ln for ln in reversed(out.splitlines()) if ln.startswith("PROBE ")), None)
    if line is None:
        return {"installed": True, "version": None, "ok": False,
                "error": f"no answer (exit {rc}): {out[-400:]}"}
    return json.loads(line[6:])


def _probe(mod):
    return _answer(_PROBE, mod)


def _pinned(mod, requires=None):
    """The exact version vLLM's own metadata requires of `mod`, or None if it pins none."""
    if requires is None:
        try:
            requires = importlib.metadata.requires("vllm") or []
        except importlib.metadata.PackageNotFoundError:
            return None
    for req in requires:
        m = re.match(r"\s*" + re.escape(mod) + r"\s*==\s*([0-9][0-9A-Za-z.]*)\s*(;.*)?$", req)
        if m and "extra" not in (m.group(2) or ""):
            return m.group(1)
    return None


def repair(mod, probe=None, run=None, pinned=None):
    """A package that is installed but cannot be imported: put back the build vLLM pins, from PyPI
    — what pip installs on a clean box. If that does not import either, remove it: a text model
    needs none of these, and transformers skips what is not installed."""
    probe, run, pinned = probe or _probe, run or _run, pinned or _pinned
    was = probe(mod)
    if not was["installed"] or was["ok"]:
        return was
    pip, steps, want = [sys.executable, "-m", "pip"], [], pinned(mod)
    if want:
        steps.append(pip + ["install", "-q", "--no-deps", "--force-reinstall", "--index-url",
                            "https://pypi.org/simple", f"{mod}=={want}"])
    steps.append(pip + ["uninstall", "-y", "-q", mod])
    now = was
    for cmd in steps:
        rc, out = run(cmd)
        now = probe(mod)
        PREFLIGHT["repairs"].append({"package": mod, "was": was, "did": " ".join(cmd[3:]),
                                     "returncode": rc, "now": now, "output_tail": out[-400:]})
        if not now["installed"] or now["ok"]:
            break
    return now


def diag_zip():
    """What an early stop leaves behind: the environment, every package's version, and whatever
    is in the results folder — the engine's whole log, if it got that far."""
    PREFLIGHT["pip_freeze"] = _run([sys.executable, "-m", "pip", "freeze"], 300)[1].splitlines()
    pathlib.Path(OUT_DIR, "preflight.json").write_text(
        json.dumps({"environment": ENV, "preflight": PREFLIGHT}, indent=1, default=str))
    with zipfile.ZipFile(f"{OUT_DIR}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(pathlib.Path(OUT_DIR).glob("*")):
            if f.suffix != ".whl":
                z.write(f)
    return f"{OUT_DIR}.zip"


# ── run it
importlib.invalidate_caches()
PREFLIGHT["torch"] = _answer(_TORCH)
print("torch      ", PREFLIGHT["torch"])
if not PREFLIGHT["torch"].get("cuda_available"):
    diag_zip()
    raise Stop("⛔ torch cannot use a GPU in this environment — the reason is above. Nothing was "
               "measured. Bring back c4_results.zip and this cell's output.")
if "T4" not in (PREFLIGHT["torch"].get("device") or ""):
    diag_zip()
    raise Stop(f"⛔ the GPU is {PREFLIGHT['torch'].get('device')!r}, not a Tesla T4. This run is registered "
               "for one T4, and its profiles are T4 profiles: nothing is measured on "
               "other hardware. Kaggle: Settings → Accelerator → GPU T4 ×2.")
PREFLIGHT["packages"] = {}
for _mod in _FAMILY:
    PREFLIGHT["packages"][_mod] = repair(_mod)
    print(f"{_mod:11s}", PREFLIGHT["packages"][_mod])
for _r in PREFLIGHT["repairs"]:
    print(f"  repaired {_r['package']}: was {_r['was'].get('version')} — "
          f"{(_r['was'].get('error') or '')[:120]}\n    did: pip {_r['did']}\n    now: {_r['now']}")
_rc, _out = _run(["vllm", "serve", "--help"], 900)
PREFLIGHT["vllm_cli"] = {"returncode": _rc, "output_tail": _out[-3000:]}
if _rc != 0:
    print(_out[-3000:])
    diag_zip()
    raise Stop("⛔ `vllm serve --help` fails in this environment, so no engine can start — the "
               "reason is above. Nothing was measured. Bring back c4_results.zip and this "
               "cell's output.")
diag_zip()
print(f"vllm        its command line starts ✓ · {len(PREFLIGHT['repairs'])} repair step(s) · "
      "pre-flight saved")


In [ ]:
# @@KVSIM-WHEEL@@ — kv-routing-sim, from its own wheel. Checked against the registered hash first.
import base64
import hashlib

_WHEEL = base64.b64decode("""...""")
if hashlib.sha256(_WHEEL).hexdigest() != "36725240737eda86aa02b9273d80eafe06727e36d39ec037d02650924831b8f0":
    raise Stop("⛔ the embedded wheel does not match its registered hash")
_path = pathlib.Path(OUT_DIR) / "kv_routing_sim-0.1.0-py3-none-any.whl"
_path.write_bytes(_WHEEL)
print(sh(f"{sys.executable} -m pip install -q {_path}", timeout=600) or "installed")
print(sh(f"{sys.executable} -m kv_routing_sim.cli calibrate --help | head -1"))


## 3 — Audit: does the installed engine contain the code the model cites?


In [ ]:
import importlib.metadata

import vllm

VLLM_VERSION = vllm.__version__
if VLLM_VERSION != VLLM_PINNED:
    raise Stop(f"⛔ vllm {VLLM_VERSION} is imported, but the predictions were made for "
               f"{VLLM_PINNED}. Restart the kernel and run from the top.")


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


ENV["packages"] = {p: _version(p) for p in ("vllm", "torch", "transformers", "triton")}
print("packages:", ENV["packages"])

ROOT = pathlib.Path(vllm.__file__).parent
SOURCES = {}
for p in ROOT.rglob("*.py"):
    try:
        SOURCES[str(p.relative_to(ROOT))] = p.read_text(errors="replace")
    except OSError:
        pass


def found_in(needle):
    return next((f for f, text in SOURCES.items() if needle in text), None)


# (what the simulator relies on, the exact source text in vLLM that implements it)
CITED = [
    ("null block reserved at start-up", "self.null_block = self.free_block_queue.popleft()"),
    ("uncached blocks reused first (prepend)", "prepend_n"),
    ("finished requests freed tail-first", "Free blocks in reverse order"),
    ("lookup capped at prompt_length - 1", "max_cache_hit_length = request.num_tokens - 1"),
    ("queries counted at admission only",
     "Record at admission so unscheduled lookups are not counted"),
    ("eviction event fired before hash check", "on_block_evicted(block)"),
    ("block access history bounded to 4", "deque(maxlen=4)"),
    ("reset also clears block metrics", "self.metrics_collector.reset()"),
    ("async scheduler caches on each output",
     "request, request.num_computed_tokens - request.num_output_placeholders"),
    ("full-prompt admission check, default on", "scheduler_reserve_full_isl: bool = True"),
    ("admission watermark, default 0", "watermark: float = Field(default=0.0"),
]
# (what this notebook sends or reads, the source text that serves it)
USED = [
    ("metric vllm:prefix_cache_queries", "vllm:prefix_cache_queries"),
    ("metric vllm:prefix_cache_hits", "vllm:prefix_cache_hits"),
    ("metric vllm:prompt_tokens_by_source", "vllm:prompt_tokens_by_source"),
    ("metric vllm:num_preemptions", "vllm:num_preemptions"),
    ("metric vllm:kv_block_lifetime_seconds", "vllm:kv_block_lifetime_seconds"),
    ("metric vllm:kv_block_idle_before_evict_seconds", "vllm:kv_block_idle_before_evict_seconds"),
    ("metric vllm:kv_block_reuse_gap_seconds", "vllm:kv_block_reuse_gap_seconds"),
    ("metric vllm:num_requests_running", "vllm:num_requests_running"),
    ("flag --num-gpu-blocks-override", "num_gpu_blocks_override"),
    ("flag --enable-prompt-tokens-details", "enable_prompt_tokens_details"),
    ("request field ignore_eos", "ignore_eos"),
    ("POST /reset_prefix_cache", '"/reset_prefix_cache"'),
    ("GET /server_info", '"/server_info"'),
]
AUDIT = {}
print(f"\nvllm {VLLM_VERSION}: {len(SOURCES):,} source files read")
for group, rows in (("the simulator cites", CITED), ("this notebook uses", USED)):
    print(f"── {group}")
    for what, needle in rows:
        AUDIT[what] = found_in(needle)
        print(f"  {'OK' if AUDIT[what] else '--'}  {what:44s} {AUDIT[what] or 'NOT FOUND'}")
AUDIT["vllm:prefix_cache_hit_rate (must NOT exist)"] = found_in('"vllm:prefix_cache_hit_rate"')
print(f"  {'OK' if not AUDIT['vllm:prefix_cache_hit_rate (must NOT exist)'] else '!!'}  "
      "vllm:prefix_cache_hit_rate does not exist (ADR-008)")

missing_used = [w for w, _ in USED if AUDIT[w] is None]
missing_cited = [w for w, _ in CITED if AUDIT[w] is None]
if missing_used:
    raise Stop(f"⛔ this vLLM lacks what the notebook needs: {missing_used}")
if missing_cited:
    print(f"\n⚠️  not found verbatim: {missing_cited}. Run anyway — the counts will show whether "
          "the behaviour itself changed.")


## 4 — Part 3's held-out traffic and the registered predictions (embedded — do not edit)


In [ ]:
# ── The traffic, regenerated — a port of kv_routing_sim/traces/generator.py (AgenticWorkload) and
#    traces/hashing.py (chain_hashes). The fingerprints prove it made the exact ids predicted on.
WORKLOADS = json.loads(r"""{"long_below": {"session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "n_sessions": 60, "turns_per_session": 4, "prefix_sharing_ratio": 0.8, "seed": 12, "arrival_rate": 0.1}, "short_at": {"session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "n_sessions": 64, "turns_per_session": 6, "prefix_sharing_ratio": 0.5, "n_shared_prefixes": 8, "prefix_zipf_s": 1.0, "seed": 11, "arrival_rate": 0.15}, "long_above": {"session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "n_sessions": 96, "turns_per_session": 4, "prefix_sharing_ratio": 0.8, "seed": 13, "arrival_rate": 0.2}}""")
FINGERPRINTS = json.loads(r"""{"long_below": "df97e84362d824b9f1481cec5d576916f85bd7ceb48fe3ca4e1f76d66d8cfd9f", "short_at": "620bf0f6c911bc2673bc56b5994e09d891e9538923636791c5fe5c19f9386be4", "long_above": "4ae26ad8396f1369bcd495614028cc9b4b874066ca9108b9d68b39838630e5d4"}""")
PRED = json.loads(r"""{"check": {"0": {"cached_tokens": [0, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272], "hit_rate": 0.714286, "hits": 9840, "queries": 13776, "requests": 48}, "1": {"cached_tokens": [0, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272], "hit_rate": 0.714286, "hits": 9840, "queries": 13776, "requests": 48}, "2": {"cached_tokens": [0, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272, 144, 208, 272], "hit_rate": 0.714286, "hits": 9840, "queries": 13776, "requests": 48}}, "decode_control": {"1408x12": {"step_s": 0.1047327189874217, "summed_context": 17328.0}, "1408x8": {"step_s": 0.08320008342882565, "summed_context": 11552.0}}, "decode_shared": {"12": {"step_s": 0.1047327189874217, "summed_context": 17328.0}, "128": {"step_s": 0.7291791501867073, "summed_context": 184832.0}, "16": {"step_s": 0.12626535454601778, "summed_context": 23104.0}, "32": {"step_s": 0.212395896780402, "summed_context": 46208.0}, "64": {"step_s": 0.3846569812491704, "summed_context": 92416.0}, "8": {"step_s": 0.08320008342882565, "summed_context": 11552.0}, "96": {"step_s": 0.5569180657179389, "summed_context": 138624.0}}, "runs": {"long_above": {"profile A": {"hit_rate": 0.922623, "itl_mean_ms": 131.865, "margin": 1.985121, "meets_per_token_limit": false, "preemptions": 0, "prefix_cache_hits": 602288, "prefix_cache_queries": 652800, "share_within_limit": 0.460938, "tpot_p50_ms": 109.683, "tpot_p95_ms": 298.512, "ttft_p50_ms": 350.149, "ttft_p95_ms": 736.046}}, "long_below": {"profile A": {"hit_rate": 0.921373, "itl_mean_ms": 64.116, "margin": 0.170997, "meets_per_token_limit": true, "preemptions": 0, "prefix_cache_hits": 375920, "prefix_cache_queries": 408000, "share_within_limit": 1.0, "tpot_p50_ms": 63.838, "tpot_p95_ms": 82.9, "ttft_p50_ms": 268.487, "ttft_p95_ms": 399.971}}, "short_at": {"profile A": {"hit_rate": 0.846544, "itl_mean_ms": 65.549, "margin": 0.042719, "meets_per_token_limit": false, "preemptions": 0, "prefix_cache_hits": 286064, "prefix_cache_queries": 337920, "share_within_limit": 0.914062, "tpot_p50_ms": 62.864, "tpot_p95_ms": 104.272, "ttft_p50_ms": 197.599, "ttft_p95_ms": 382.254}}}, "stall": {"32+128": {"extra_gap_s": 0.18443932037300104, "prefill_s": 0.17890236860427608, "usual_gap_s": 0.2137380891796099}, "64+128": {"extra_gap_s": 0.18443932037300098, "prefill_s": 0.17890236860427608, "usual_gap_s": 0.38734136604758623}, "8+128": {"extra_gap_s": 0.18443932037300098, "prefill_s": 0.17890236860427608, "usual_gap_s": 0.08353563152862764}}}""")
PREDICTIONS_SHA256 = "818e2c9522231f9ea1711ac06a9d4be0ae44b9221059a12b77f609caeba7bde5"

_SHARED_BASE, _PREFIX_STRIDE, _SESSION_BASE, _TURN_BASE = 0, 100_000, 10_000_000, 500_000_000
_MIX_MUL, _MIX_ADD, _MASK64 = 0x9E3779B97F4A7C15, 0x632BE59BD9B4E019, 0xFFFFFFFFFFFFFFFF
_LOW, _HIGH = 1000, 100_000  # ENGINE_SAFE_IDS: ordinary text tokens in Qwen3's vocabulary


def _engine_id(raw):
    return _LOW + ((raw * _MIX_MUL + _MIX_ADD) & _MASK64) % (_HIGH - _LOW)


def generate(w):
    """AgenticWorkload._iter_with_tokens + prefix_assignment, sorted by arrival like generate()."""
    turns, ctx = w["turns_per_session"], w["session_context_tokens"]
    user_n, asst_n, r = w["user_turn_tokens"], w["assistant_turn_tokens"], w["prefix_sharing_ratio"]
    mean_private = ctx + user_n + (user_n + asst_n) * (turns - 1) / 2.0
    shared_len = 0 if r <= 0.0 else max(0, round(r / (1.0 - r) * mean_private))
    k_total = w.get("n_shared_prefixes", 1)
    prefixes = [[_SHARED_BASE + k * _PREFIX_STRIDE + i for i in range(shared_len)]
                for k in range(k_total)]
    if k_total == 1:
        assign = [0] * w["n_sessions"]
    else:
        ks = range(k_total)
        weights = [1.0 / (k + 1) ** w.get("prefix_zipf_s", 1.0) for k in ks]
        assign = random.Random(f"shared-prefix-{w['seed']}").choices(ks, weights=weights,
                                                                     k=w["n_sessions"])
    rng, clock, out = random.Random(w["seed"]), 0.0, []
    for s in range(w["n_sessions"]):
        clock += rng.expovariate(w["arrival_rate"])
        shared = prefixes[assign[s]]
        private = [_SESSION_BASE + s * 100_000 + i for i in range(ctx)]
        history = []
        for t in range(turns):
            base = _TURN_BASE + s * 100_000 + t * 1_000
            user = [base + i for i in range(user_n)]
            out.append({"request_id": f"s{s}-t{t}",
                        "arrival_ts": round(clock + t * w["think_time_s"], 6),
                        "session_id": f"s{s}",
                        "ids": [_engine_id(x) for x in shared + private + history + user]})
            history = history + user + [base + 500 + i for i in range(asst_n)]
    out.sort(key=lambda q: q["arrival_ts"])
    return out


def _h64(data):
    return int.from_bytes(hashlib.blake2b(data, digest_size=8).digest(), "big")


def chain_hashes(ids, block_size=BLOCK_SIZE, salt=0):
    """traces/hashing.py: one hash per FULL block, each chained to its parent."""
    out, parent = [], _h64(salt.to_bytes(8, "big"))
    for i in range(len(ids) // block_size):
        body = b"".join(int(t).to_bytes(8, "big", signed=True)
                        for t in ids[i * block_size:(i + 1) * block_size])
        parent = _h64(parent.to_bytes(8, "big") + body)
        out.append(parent)
    return out


def _canonical(obj):
    return json.dumps(obj, separators=(",", ":"), sort_keys=True).encode()


TRACES = {}
for _name, _w in WORKLOADS.items():
    _tr = generate(_w)
    _fp = hashlib.sha256(_canonical([[q["request_id"], q["arrival_ts"], q["session_id"], q["ids"]]
                                     for q in _tr])).hexdigest()
    if _fp != FINGERPRINTS[_name]:
        raise Stop(f"⛔ {_name}: the regenerated trace does not match its fingerprint — this "
                   "Python's random module or the generator port differs from the one predicted on")
    for _q in _tr:
        _q["blocks"] = chain_hashes(_q["ids"])
    TRACES[_name] = _tr
    print(f"{_name:4s} ✓  {len(_tr)} requests · prompts {min(len(q['ids']) for q in _tr)}–"
          f"{max(len(q['ids']) for q in _tr)} tokens · fingerprint {_fp[:16]}…")
if hashlib.sha256(_canonical(PRED)).hexdigest() != PREDICTIONS_SHA256:
    raise Stop("⛔ the embedded predictions do not match their hash: the data cell was edited")
print(f"predictions ✓ {PREDICTIONS_SHA256[:16]}…\n\nRegistered for profile A, one replica (hit rate; the rest is in PRED):")
for _name, _by in PRED["runs"].items():
    print(f"  {_name:4s} " + "   ".join(f"{p} {v['hit_rate']:.4f}" for p, v in _by.items()))


## 5 — One server


In [ ]:
SERVERS = {}  # replica -> {"proc", "logf", "log_path"}
RESET_MODE = "endpoint"  # falls back to "restart" if the dev endpoint is not served


def engine_argv(i):
    return [
        "vllm", "serve", MODEL, "--port", str(PORTS[i]),
        "--dtype", "half",  # T4 has no bf16
        "--block-size", str(BLOCK_SIZE),
        "--num-gpu-blocks-override", str(NUM_GPU_BLOCKS),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--enable-prefix-caching",
        "--enable-prompt-tokens-details",
        "--seed", "0",
    ]


def port_in_use(port):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0


def start_server(i):
    # ⚠️ Refuse to launch while ANYTHING listens on the port: a stale server would answer /health
    # before ours had time to die of "address already in use" (found by Phase 3's dry-run).
    for _ in range(30):
        if not port_in_use(PORTS[i]):
            break
        time.sleep(1)
    else:
        raise Stop(f"⛔ something already serves on port {PORTS[i]} — a server from an earlier "
                   "run? Run the clean-up cell (or !pkill -f 'vllm serve') and start again.")
    # One physical GPU per replica: complete isolation, so TTFT is attributable (ADR-005).
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), VLLM_SERVER_DEV_MODE="1")
    log_path = pathlib.Path(OUT_DIR) / f"server_{i}.log"
    logf = open(log_path, "w")  # noqa: SIM115 - must outlive this cell
    proc = subprocess.Popen(engine_argv(i), stdout=logf, stderr=subprocess.STDOUT, env=env)
    SERVERS[i] = {"proc": proc, "logf": logf, "log_path": str(log_path)}
    print(f"launched replica {i}: GPU {i}, port {PORTS[i]}")


def log_text(i):
    return pathlib.Path(SERVERS[i]["log_path"]).read_text(errors="replace") if i in SERVERS else ""


def http(i, method, path, payload=None, timeout=60):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[i]}{path}", data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")
    except Exception as e:
        return 0, repr(e)


def wait_ready(i, timeout=STARTUP_TIMEOUT_S, every=30):
    """Ready = OUR process is alive AND the port answers, checked in that order."""
    t0, last = time.time(), 0.0
    while time.time() - t0 < timeout:
        if SERVERS[i]["proc"].poll() is not None:
            print(f"  !! replica {i} EXITED (returncode {SERVERS[i]['proc'].returncode}). Log tail:")
            print(log_text(i)[-3000:])
            return False
        if http(i, "GET", "/health", timeout=5)[0] == 200:
            print(f"  replica {i} ready in {time.time() - t0:.0f}s")
            return True
        if time.time() - t0 - last >= every:
            last = time.time() - t0
            tail = log_text(i).strip().splitlines()
            print(f"    ...{last:.0f}s  {tail[-1][:110] if tail else '(log empty)'}")
        time.sleep(5)
    print(f"  !! replica {i} not ready within {timeout}s")
    return False


def stop_server(i):
    if i not in SERVERS:
        return
    SERVERS[i]["proc"].terminate()
    try:
        SERVERS[i]["proc"].wait(timeout=90)
    except subprocess.TimeoutExpired:
        SERVERS[i]["proc"].kill()
    SERVERS[i]["logf"].close()
    del SERVERS[i]


def stop_all():
    for i in list(SERVERS):
        stop_server(i)


def metric(text, family, **labels):
    """Sum a family over matching labels. None = ABSENT, which is not the same as 0."""
    lab = "".join(f'(?=[^}}]*{k}="{v}")' for k, v in labels.items())
    pat = re.compile(r"^" + re.escape(family) + r"(?:_total)?\{" + lab + r"[^}]*\}\s+(\S+)",
                     re.MULTILINE)
    vals = [float(x) for x in pat.findall(text)]
    return sum(vals) if vals else None


COUNTERS = {  # our name -> (metric family, labels)
    "prefix_cache_queries": ("vllm:prefix_cache_queries", {}),
    "prefix_cache_hits": ("vllm:prefix_cache_hits", {}),
    "local_compute": ("vllm:prompt_tokens_by_source", {"source": "local_compute"}),
    "local_cache_hit": ("vllm:prompt_tokens_by_source", {"source": "local_cache_hit"}),
    "preemptions": ("vllm:num_preemptions", {}),
    "prompt_tokens": ("vllm:prompt_tokens", {}),
    "generation_tokens": ("vllm:generation_tokens", {}),
}


def read_counters(text):
    return {k: metric(text, fam, **lab) for k, (fam, lab) in COUNTERS.items()}


def settled_scrape(i, max_wait=300):
    """Scrape once nothing runs or waits AND the counters stopped moving: stats reach /metrics a
    moment after the tokens they describe, so an early read under-counts."""
    t0, prev = time.time(), None
    while True:
        text = http(i, "GET", "/metrics", timeout=30)[1]
        busy = ((metric(text, "vllm:num_requests_running") or 0)
                + (metric(text, "vllm:num_requests_waiting") or 0))
        now = read_counters(text)
        if busy == 0 and now == prev:
            return text
        if time.time() - t0 > max_wait:
            raise Stop(f"⛔ replica {i} never settled within {max_wait}s (busy={busy})")
        prev = now
        time.sleep(1.0)


def deltas(before, after):
    b, a = read_counters(before), read_counters(after)
    return {k: None if a[k] is None else round(a[k] - (b[k] or 0.0)) for k in COUNTERS}


def fresh_caches():
    """A cold cache on BOTH replicas before every run, by /reset_prefix_cache (a restart if the
    endpoint is not served)."""
    global RESET_MODE
    for i in range(N_REPLICAS):
        if RESET_MODE == "endpoint":
            for _ in range(15):
                code, body = http(i, "POST", "/reset_prefix_cache")
                try:
                    ok = code == 200 and json.loads(body).get("success") is True
                except ValueError:
                    ok = False
                if ok:
                    break
                if code in (404, 405):
                    print("  /reset_prefix_cache is not served: restarting the engines per run")
                    RESET_MODE = "restart"
                    break
                time.sleep(2)  # success=false: a finishing request still holds blocks
            else:
                raise Stop(f"⛔ replica {i} refused to reset its prefix cache 15 times")
        if RESET_MODE == "restart":
            stop_server(i)
            start_server(i)
            if not wait_ready(i):
                raise Stop(f"⛔ replica {i} failed to restart — the log tail is above")


print("server helpers ready")


In [ ]:
CAPACITY = {}
for _i in range(N_REPLICAS):  # one at a time: the first start downloads the weights for both
    start_server(_i)
    if not wait_ready(_i):
        diag_zip()
        raise Stop(f"⛔ replica {_i} did not start — the log tail is above. Nothing was measured. "
                   "Bring back c4_results.zip: it holds the whole log.")
    _log = log_text(_i)
    _m = re.search(r"KV cache size: ([\d,]+) tokens", _log)
    CAPACITY[_i] = {
        "kv_cache_size_tokens_logged": int(_m.group(1).replace(",", "")) if _m else None,
        "expected_tokens": NUM_GPU_BLOCKS * BLOCK_SIZE,
        "override_logged": f"num_gpu_blocks_override={NUM_GPU_BLOCKS}" in _log,
    }
    print(f"  replica {_i}: {CAPACITY[_i]}")
    if CAPACITY[_i]["kv_cache_size_tokens_logged"] != CAPACITY[_i]["expected_tokens"]:
        diag_zip()
        raise Stop(f"⛔ replica {_i} logged {CAPACITY[_i]['kv_cache_size_tokens_logged']} tokens "
                   f"of KV cache, not {CAPACITY[_i]['expected_tokens']:,}: every prediction "
                   f"assumes exactly {NUM_GPU_BLOCKS} blocks per replica.")


## 6 — The client, the command, the decode cells, the held-out run, and saving


In [ ]:
from statistics import median


def stream_one(i, ids, max_tokens):
    """One streamed completion to replica i: send time, the arrival time of EVERY chunk (one chunk
    per token on vLLM 0.30 — checked in the warm-up below), and the usage block."""
    body = {"model": MODEL, "prompt": ids, "max_tokens": max_tokens, "temperature": 0.0,
            "ignore_eos": True, "stream": True, "stream_options": {"include_usage": True}}
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[i]}/v1/completions",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    rec = {"t_send": time.perf_counter(), "t_first": None, "t_chunks": [], "usage": None,
           "finish_reason": None, "error": None}
    try:
        with urllib.request.urlopen(req, timeout=1800) as resp:
            for raw in resp:
                line = raw.strip()
                if not line.startswith(b"data:"):
                    continue
                data = line[5:].strip()
                if data == b"[DONE]":
                    break
                obj = json.loads(data)
                if obj.get("choices"):
                    t = time.perf_counter()
                    rec["t_chunks"].append(t)
                    if rec["t_first"] is None:
                        rec["t_first"] = t
                    reason = obj["choices"][0].get("finish_reason")
                    if reason:
                        rec["finish_reason"] = reason
                if obj.get("usage"):
                    rec["usage"] = obj["usage"]
    except Exception as e:
        rec["error"] = repr(e)
    rec["t_done"] = time.perf_counter()
    return rec


def rand_ids(rng, n):
    """Ordinary text tokens, random: two prompts share no 16-token block by chance."""
    return [rng.randrange(1000, 100_000) for _ in range(n)]


def usable(rec, prompt_len, out):
    u = rec["usage"] or {}
    return (rec["error"] is None and rec["finish_reason"] == "length"
            and u.get("completion_tokens") == out and u.get("prompt_tokens") == prompt_len)


def itl_totals(text):
    return (metric(text, "vllm:inter_token_latency_seconds_sum"),
            metric(text, "vllm:inter_token_latency_seconds_count"))


def itl_mean(before, after):
    """Engine-side mean inter-token latency between two scrapes (vLLM measures it itself)."""
    (s0, c0), (s1, c1) = itl_totals(before), itl_totals(after)
    if None in (s0, c0, s1, c1) or c1 - c0 <= 0:
        return None
    return (s1 - s0) / (c1 - c0)


def reset_replica(i):
    """A cold cache on ONE replica (the other may be busy in part C). Timing needs no reset —
    prompts are random — so a refusal is recorded, never fatal."""
    for _ in range(15):
        code, body = http(i, "POST", "/reset_prefix_cache")
        try:
            if code == 200 and json.loads(body).get("success") is True:
                return True
        except ValueError:
            pass
        if code in (404, 405):
            return False
        time.sleep(2)
    return False


def nearest_rank(sorted_vals, q):
    if not sorted_vals:
        return None
    return sorted_vals[max(0, min(len(sorted_vals) - 1, round(q * (len(sorted_vals) - 1))))]


def measure_decode(i, L, b, seed):
    """b requests of L random tokens at once. A request's decode step = mean gap from token `skip`
    to its last; the cell's = the median over requests. Invalid if any request failed or anything
    was preempted — a preempted request's gaps would time recomputation, not decode."""
    out, skip = DESIGN["decode"]["output_tokens"], DESIGN["decode"]["skip_tokens"]
    rng = random.Random(seed)
    prompts = [rand_ids(rng, L) for _ in range(b)]
    was_reset = reset_replica(i)
    before = settled_scrape(i)
    with ThreadPoolExecutor(max_workers=b) as pool:
        recs = list(pool.map(lambda ids: stream_one(i, ids, out), prompts))
    after = settled_scrape(i)
    d = deltas(before, after)
    steps = []
    for r in recs:
        t = r["t_chunks"]
        if usable(r, L, out) and len(t) == out:
            steps.append((t[-1] - t[skip]) / (out - 1 - skip))
    return {
        "L": L, "b": b, "seed": seed, "reset": was_reset,
        "valid": len(steps) == b and (d["preemptions"] or 0) == 0,
        "step_s": median(steps) if steps else None,
        "request_steps_s": [round(x, 6) for x in steps],
        "ttft_s": [round(r["t_first"] - r["t_send"], 6) for r in recs if r["t_first"]],
        "server_itl_s": itl_mean(before, after),
        "preemptions": d["preemptions"], "failed": b - len(steps),
    }


def run_calibrate(seed):
    """`kvsim calibrate` as a user runs it, from the embedded wheel: its own process, every timing
    cell at its default. Never raises — a failure is a result, and part 2 must still run."""
    out = pathlib.Path(OUT_DIR) / f"calibrated_profile_seed{seed}.json"
    out.unlink(missing_ok=True)
    cmd = [sys.executable, "-m", "kv_routing_sim.cli", "calibrate",
           "--engine", f"http://127.0.0.1:{PORTS[MEASURE]}", "--model", MODEL, "--out", str(out),
           "--seed", str(seed), "--check-sessions", str(CAL["check_sessions"]),
           "--check-turns", str(CAL["check_turns"]), "--max-error", str(CAL["max_error"]),
           "--token-range", f"{CAL['token_range'][0]},{CAL['token_range'][1]}"]
    print("$", " ".join(cmd))
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600, check=False)
        rc, stdout, stderr = proc.returncode, proc.stdout, proc.stderr
    except subprocess.TimeoutExpired as e:
        rc, stdout, stderr = None, str(e.stdout or ""), f"timed out after {e.timeout} s"
    wall = round(time.time() - t0, 1)
    pathlib.Path(OUT_DIR, f"calibrate_seed{seed}_stdout.txt").write_text(stdout)
    pathlib.Path(OUT_DIR, f"calibrate_seed{seed}_stderr.txt").write_text(stderr)
    print(stdout)
    print(stderr)
    profile = json.loads(out.read_text()) if out.exists() else None
    chk = ((profile or {}).get("_source") or {}).get("check") or {}
    reg, pred = PRED["check"][str(seed)], chk.get("predicted") or {}
    rec = {
        "seed": seed, "returncode": rc, "wall_clock_s": wall,
        "profile_file": out.name if profile else None,
        "calibrated": bool(profile and profile.get("calibrated")),
        "read_blocks": profile and profile.get("kv_cache_blocks"),
        "check_measured": chk.get("measured"), "check_predicted": pred,
        "cached_tokens_equal": chk.get("cached_tokens_equal"),
        "cached_tokens_compared": chk.get("cached_tokens_compared"),
        # The embedded code must predict exactly the registered numbers; if not, the wheel is not
        # the code that made the prediction, and nothing here can be compared.
        "prediction_matches_registration": (pred.get("hits"), pred.get("queries"))
        == (reg["hits"], reg["queries"]),
    }
    print(f"exit code {rc} · {wall} s · calibrated {rec['calibrated']} · check requests equal "
          f"{rec['cached_tokens_equal']}/{rec['cached_tokens_compared']} (registered "
          f"{reg['requests']}) · blocks read {rec['read_blocks']}")
    return rec


import zipfile

TPOT_FAMILY = "vllm:request_time_per_output_token_seconds"
ITL_FAMILY = "vllm:inter_token_latency_seconds"


def hist(text, family):
    """A histogram family from /metrics: cumulative counts by upper bound, and sum and count."""
    pat = re.compile(r"^" + re.escape(family) + r'_bucket\{[^}]*le="([^"]+)"[^}]*\}\s+(\S+)',
                     re.MULTILINE)
    buckets = {}
    for le, v in pat.findall(text):
        buckets[le] = buckets.get(le, 0.0) + float(v)
    return {"buckets": buckets, "sum": metric(text, family + "_sum"),
            "count": metric(text, family + "_count")}


def hist_delta(before, after, family):
    """What a histogram family gained between two scrapes. None if the engine does not have it."""
    b, a = hist(before, family), hist(after, family)
    if not a["buckets"] or a["count"] is None:
        return None
    return {"buckets": {le: v - b["buckets"].get(le, 0.0) for le, v in a["buckets"].items()},
            "sum": a["sum"] - (b["sum"] or 0.0), "count": a["count"] - (b["count"] or 0.0)}


def share_within(h, limit_s):
    """The share of a histogram's observations at or under `limit_s` — exact when `limit_s` is one
    of its bucket bounds, None when it is not (no interpolation: that would invent a number)."""
    if not h or not h["count"]:
        return None
    for le, v in h["buckets"].items():
        if le != "+Inf" and abs(float(le) - limit_s) < 1e-12:
            return v / h["count"]
    return None


def measure_decode_shared(i, prefix_len, suffix_len, b, seed):
    """b requests that SHARE a cached prefix — the prefix, then a unique suffix each — sent together.

    The prefix is cached by one request first, so every one of the b finds it. Its blocks are then
    stored once, however many sequences use them: far more sequences fit than with unique prompts,
    while each still attends over its whole context. That is the regime a shared system prompt
    puts an engine in, and the one no unique-prompt cell can reach. Timed as part A of C2 was: a
    request's decode step = the mean gap from token `skip` to its last; the cell's = the median.
    Invalid if any request failed, anything was preempted, or a request did not hit the prefix."""
    out, skip = DESIGN["decode"]["output_tokens"], DESIGN["decode"]["skip_tokens"]
    rng = random.Random(seed)
    prefix = rand_ids(rng, prefix_len)
    prompts = [prefix + rand_ids(rng, suffix_len) for _ in range(b)]
    was_reset = reset_replica(i)
    settled_scrape(i)
    warm = stream_one(i, prefix, 1)
    if not usable(warm, prefix_len, 1):
        return {"prefix": prefix_len, "suffix": suffix_len, "b": b, "seed": seed, "valid": False,
                "step_s": None, "error": warm["error"]}
    before = settled_scrape(i)
    with ThreadPoolExecutor(max_workers=b) as pool:
        recs = list(pool.map(lambda ids: stream_one(i, ids, out), prompts))
    after = settled_scrape(i)
    d = deltas(before, after)
    steps, cached = [], []
    for r in recs:
        t = r["t_chunks"]
        if usable(r, prefix_len + suffix_len, out) and len(t) == out:
            steps.append((t[-1] - t[skip]) / (out - 1 - skip))
            cached.append(((r["usage"] or {}).get("prompt_tokens_details") or {})
                          .get("cached_tokens"))
    want = (prefix_len // BLOCK_SIZE) * BLOCK_SIZE
    return {
        "prefix": prefix_len, "suffix": suffix_len, "b": b, "seed": seed, "reset": was_reset,
        "valid": (len(steps) == b and (d["preemptions"] or 0) == 0
                  and all(c == want for c in cached)),
        "step_s": median(steps) if steps else None,
        "request_steps_s": [round(x, 6) for x in steps],
        "cached_tokens_seen": sorted({str(c) for c in cached}), "cached_tokens_expected": want,
        "server_itl_s": itl_mean(before, after),
        "preemptions": d["preemptions"], "failed": b - len(steps),
    }


def measure_stall(i, prefix_len, suffix_len, b, new_tokens, delay_s, seed):
    """What ONE arriving request's prefill costs the sequences that are already decoding.

    b requests that share a cached prefix start decoding. `delay_s` later one more request arrives:
    the same prefix, then `new_tokens` unique tokens. The step that prefills it also carries
    everyone else's next token — so each running sequence sees one long gap. The simulator's rule
    is that this step costs prefill + decode, added; no cell has measured that until now.

    Per running sequence: the gap that ends when the late request's first token arrives, and the
    median of its other gaps. Invalid if a request failed, anything was preempted, the late request
    missed the prefix, or it did not land while the others were mid-decode."""
    out, skip = DESIGN["stall"]["output_tokens"], DESIGN["decode"]["skip_tokens"]
    rng = random.Random(seed)
    prefix = rand_ids(rng, prefix_len)
    prompts = [prefix + rand_ids(rng, suffix_len) for _ in range(b)]
    late = prefix + rand_ids(rng, new_tokens)
    was_reset = reset_replica(i)
    settled_scrape(i)
    warm = stream_one(i, prefix, 1)
    if not usable(warm, prefix_len, 1):
        return {"b": b, "new_tokens": new_tokens, "seed": seed, "valid": False,
                "error": warm["error"]}
    before = settled_scrape(i)
    with ThreadPoolExecutor(max_workers=b + 1) as pool:
        futs = [pool.submit(stream_one, i, ids, out) for ids in prompts]
        time.sleep(delay_s)
        late_fut = pool.submit(stream_one, i, late, 1)
        recs, late_rec = [f.result() for f in futs], late_fut.result()
    after = settled_scrape(i)
    d = deltas(before, after)
    want = (prefix_len // BLOCK_SIZE) * BLOCK_SIZE
    late_cached = ((late_rec["usage"] or {}).get("prompt_tokens_details") or {}).get(
        "cached_tokens")
    t_late = late_rec["t_first"]
    at, usual, position = [], [], []
    for r in recs:
        t = r["t_chunks"]
        if not (usable(r, prefix_len + suffix_len, out) and len(t) == out and t_late):
            continue
        # The token of this sequence that arrived with the late request's first token: the one
        # whose arrival time is nearest to it.
        k = min(range(1, out), key=lambda j: abs(t[j] - t_late))
        gaps = [t[j] - t[j - 1] for j in range(max(skip, 1), out) if j != k]
        at.append(t[k] - t[k - 1])
        usual.append(median(gaps))
        position.append(k)
    ok = (len(at) == b and (d["preemptions"] or 0) == 0 and usable(late_rec, len(late), 1)
          and late_cached == want and all(skip < k < out - 4 for k in position))
    return {
        "prefix": prefix_len, "suffix": suffix_len, "b": b, "new_tokens": new_tokens,
        "delay_s": delay_s, "seed": seed, "reset": was_reset, "valid": ok,
        "gap_at_arrival_s": median(at) if at else None,
        "usual_gap_s": median(usual) if usual else None,
        "token_position": [min(position), max(position)] if position else None,
        "late_ttft_s": None if t_late is None else t_late - late_rec["t_send"],
        "late_cached_tokens": late_cached, "preemptions": d["preemptions"],
        "failed": b - len(at),
    }


def run_holdout(name):
    """Held-out traffic on ONE engine, open-loop at its arrival times — C2's part D, and now
    every request's token times are kept: its first token, its last, and how many came between.
    The engine's own two per-token histograms are read before and after."""
    trace, out = TRACES[name], WORKLOADS[name]["output_tokens"]
    reset_replica(MEASURE)
    before = settled_scrape(MEASURE)
    t_zero = time.perf_counter() + 2.0
    with ThreadPoolExecutor(max_workers=len(trace)) as pool:
        futs = []
        for q in trace:
            delay = t_zero + q["arrival_ts"] - time.perf_counter()
            if delay > 0:
                time.sleep(delay)
            futs.append((q, pool.submit(stream_one, MEASURE, q["ids"], out)))
        results = [(q, f.result()) for q, f in futs]
    after = settled_scrape(MEASURE, max_wait=900)
    d = deltas(before, after)
    recs = []
    for q, rec in results:
        u, t = rec["usage"] or {}, rec["t_chunks"]
        ok = usable(rec, len(q["ids"]), out)
        recs.append({
            "request_id": q["request_id"], "valid": ok,
            "cached_tokens": (u.get("prompt_tokens_details") or {}).get("cached_tokens"),
            "ttft_ms": None if rec["t_first"] is None else (rec["t_first"] - rec["t_send"]) * 1e3,
            # Time per output token, as MLPerf and vLLM define it: (last − first) ÷ (tokens − 1).
            # Only when every token came as its own chunk — else the count of gaps is not known.
            "tpot_ms": (t[-1] - t[0]) / (len(t) - 1) * 1e3 if ok and len(t) == out > 1 else None,
            "chunks": len(t),
            "send_offset_s": rec["t_send"] - t_zero, "scheduled_offset_s": q["arrival_ts"],
        })
    ttfts = sorted(r["ttft_ms"] for r in recs if r["ttft_ms"] is not None)
    tpots = sorted(r["tpot_ms"] for r in recs if r["tpot_ms"] is not None)
    hits, queries = d["prefix_cache_hits"] or 0, d["prefix_cache_queries"] or 0
    tpot_hist = hist_delta(before, after, TPOT_FAMILY)
    run = {
        "workload": name, "counters": d, "hit_rate": hits / queries if queries else None,
        "preemptions": d["preemptions"], "server_itl_s": itl_mean(before, after),
        "ttft_p50_ms": nearest_rank(ttfts, 0.50), "ttft_p95_ms": nearest_rank(ttfts, 0.95),
        "tpot_p50_ms": nearest_rank(tpots, 0.50), "tpot_p95_ms": nearest_rank(tpots, 0.95),
        "client_share_within_limit": (sum(1 for x in tpots if x <= TPOT_LIMIT_S * 1e3)
                                      / len(tpots) if tpots else None),
        "engine_tpot_hist": tpot_hist, "engine_itl_hist": hist_delta(before, after, ITL_FAMILY),
        "engine_share_within_limit": share_within(tpot_hist, TPOT_LIMIT_S),
        "invalid": sum(1 for r in recs if not r["valid"]),
        "without_tpot": sum(1 for r in recs if r["tpot_ms"] is None),
        "worst_send_lag_ms": max(r["send_offset_s"] - r["scheduled_offset_s"] for r in recs) * 1e3,
        "requests": recs,
    }
    pred = PRED["runs"][name]["profile A"]
    share = run["engine_share_within_limit"]
    print(f"  {name:10s} hit {run['hit_rate']:.4f} ({pred['hit_rate']:.4f}) · TTFT p95 "
          f"{run['ttft_p95_ms'] or 0:.0f} ms ({pred['ttft_p95_ms']:.0f}) · ms per token p50 "
          f"{run['tpot_p50_ms'] or 0:.1f}, p95 {run['tpot_p95_ms'] or 0:.1f} "
          f"({pred['tpot_p95_ms']:.1f}) · at or under {TPOT_LIMIT_S * 1e3:.0f} ms: client "
          f"{(run['client_share_within_limit'] or 0):.1%}, engine "
          f"{'—' if share is None else f'{share:.1%}'} ({pred['share_within_limit']:.1%}) · "
          f"preemptions {run['preemptions']} · {run['invalid']} invalid · send lag ≤ "
          f"{run['worst_send_lag_ms']:.0f} ms     [in brackets: registered for profile A]")
    return run


RESULTS = {"notebook": "notebooks/c4_per_token_t4.ipynb", "design": DESIGN,
           "fingerprints": FINGERPRINTS, "predictions_sha256": PREDICTIONS_SHA256,
           "wheel_sha256": WHEEL_SHA256, "tpot_limit_s": TPOT_LIMIT_S,
           "calibrate": [], "decode_shared": [], "decode_control": [], "stall": [],
           "holdout": []}


def save():
    """Write everything measured so far, and the zip: a later failure keeps the earlier parts."""
    RESULTS.update({"environment": ENV, "preflight": PREFLIGHT, "metric_and_source_audit": AUDIT,
                    "capacity": CAPACITY,
                    "engine_argv": [engine_argv(i) for i in range(N_REPLICAS)]})
    pathlib.Path(OUT_DIR, "c4.json").write_text(json.dumps(RESULTS, indent=1, default=str))
    with zipfile.ZipFile(f"{OUT_DIR}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(pathlib.Path(OUT_DIR).glob("*")):
            if f.suffix != ".whl":  # the wheel is in the notebook already
                z.write(f)
    return f"{OUT_DIR}.zip"


print("client, the command, the decode cells, the held-out run and saving ready")


In [ ]:
# Throwaway requests on ids that share nothing with the traces (every run below starts from a reset)
# — and a check, before any measured run, that each response reports what the comparison needs.
for _i in range(N_REPLICAS):
    _ids = [1000 + 7 * j + _i for j in range(200)]
    _rec = stream_one(_i, _ids, 8)
    _u = _rec["usage"] or {}
    if (_rec["error"] or _rec["finish_reason"] != "length" or _u.get("completion_tokens") != 8
            or (_u.get("prompt_tokens_details") or {}).get("cached_tokens") is None):
        raise Stop(f"⛔ replica {_i}'s warm-up response is unusable — is ignore_eos honoured, is "
                   f"--enable-prompt-tokens-details in effect? {_rec}")
    _absent = [k for k, v in read_counters(http(_i, "GET", "/metrics")[1]).items() if v is None]
    if _absent:
        raise Stop(f"⛔ replica {_i}'s /metrics lacks counters the comparison reads: {_absent}")
print("warm-up done: both replicas report cached_tokens, exact lengths and every counter")


In [ ]:
# The design reads decode steps from chunk arrival times: that needs one chunk per token.
_rec = stream_one(MEASURE, [1000 + j for j in range(64)], 16)
if not usable(_rec, 64, 16) or len(_rec["t_chunks"]) != 16:
    raise Stop(f"⛔ 16 tokens arrived in {len(_rec['t_chunks'])} chunks: part A's step timing "
               "assumes one chunk per token")
_itl = itl_totals(http(MEASURE, "GET", "/metrics")[1])
if None in _itl:
    print("⚠️  vllm:inter_token_latency_seconds is absent: part C's engine-side test cannot be "
          "decided (the client-side steps are still recorded)")
print(f"one chunk per token ✓ · engine-side inter-token latency {'✓' if None not in _itl else '—'}")


In [ ]:
# Part 3 reads the engine's own per-request histogram, and its verdict on the per-token limit
# needs a bucket bound at exactly that limit. Both are checked now, on the engine as it runs.
_h = hist(http(MEASURE, "GET", "/metrics")[1], TPOT_FAMILY)
_bounds = sorted(float(le) for le in _h["buckets"] if le != "+Inf")
RESULTS["tpot_histogram_bounds"] = _bounds
if not _bounds:
    print(f"⚠️  {TPOT_FAMILY} is absent: the engine-side share cannot be read "
          "(every request's client-side token times are still recorded)")
elif not any(abs(b - TPOT_LIMIT_S) < 1e-12 for b in _bounds):
    print(f"⚠️  no bucket bound at {TPOT_LIMIT_S} s (bounds: {_bounds}): the engine-side share "
          "cannot be read exactly, and is not interpolated")
else:
    print(f"{TPOT_FAMILY}: {len(_bounds)} bounds, one at {TPOT_LIMIT_S} s ✓")


## 7 — Part 1: `kvsim calibrate`, with its repeats


In [ ]:
RESULTS["calibrate"].append(run_calibrate(CAL["seeds"][0]))
print(f"saved {save()}")


## 8 — Part 2: decode steps of batches that share a cached prefix


In [ ]:
_s = DESIGN["decode_shared"]
for _rep in range(_s["reps"]):
    for _L, _b in _s["controls"]:
        _m = measure_decode(MEASURE, _L, _b, seed=f"ctl-{_L}-{_b}-{_rep}")
        RESULTS["decode_control"].append({"rep": _rep, **_m})
        print(f"  control rep {_rep}  unique prompts  L {_L:>5}  b {_b:>3}  step "
              f"{(_m['step_s'] or 0) * 1e3:7.1f} ms  {'✓' if _m['valid'] else '⛔ invalid'}")
    for _b in _s["batches"]:
        _m = measure_decode_shared(MEASURE, _s["prefix_tokens"], _s["suffix_tokens"], _b,
                                   seed=f"shared-{_b}-{_rep}")
        RESULTS["decode_shared"].append({"rep": _rep, **_m})
        print(f"  shared  rep {_rep}  prefix {_s['prefix_tokens']} + {_s['suffix_tokens']}  "
              f"b {_b:>3}  step {(_m['step_s'] or 0) * 1e3:7.1f} ms  preemptions "
              f"{_m.get('preemptions')}  cached {_m.get('cached_tokens_seen')}  "
              f"{'✓' if _m['valid'] else '⛔ invalid'}")
    print(f"saved {save()}")


## 8b — Part 2b: what one arriving request costs the sequences already decoding


In [ ]:
_t = DESIGN["stall"]
for _rep in range(_t["reps"]):
    for _b, _n, _delay in _t["cells"]:
        _m = measure_stall(MEASURE, _s["prefix_tokens"], _s["suffix_tokens"], _b, _n, _delay,
                           seed=f"stall-{_b}-{_n}-{_rep}")
        RESULTS["stall"].append({"rep": _rep, **_m})
        print(f"  stall rep {_rep}  b {_b:>3}  + {_n} new tokens after {_delay} s:  gap "
              f"{(_m.get('gap_at_arrival_s') or 0) * 1e3:7.1f} ms against the usual "
              f"{(_m.get('usual_gap_s') or 0) * 1e3:6.1f} ms  at token {_m.get('token_position')}"
              f"  late TTFT {(_m.get('late_ttft_s') or 0) * 1e3:6.1f} ms  "
              f"{'✓' if _m['valid'] else '⛔ invalid'}")
print(f"saved {save()}")


## 9 — Part 3: held-out traffic around the per-token limit


In [ ]:
for _name in WORKLOADS:
    RESULTS["holdout"].append(run_holdout(_name))
    print(f"saved {save()}")


## 10 — Part 4: `kvsim calibrate` again, after the traffic


In [ ]:
for _seed in CAL["seeds"][1:]:
    RESULTS["calibrate"].append(run_calibrate(_seed))
    print(f"saved {save()}")
_zip = save()
print(f"\nsaved {_zip} ({pathlib.Path(_zip).stat().st_size / 1e6:.1f} MB)")
print("⭐ c4_results.zip is complete — then the clean-up cell.")


## 11 — Clean-up


In [ ]:
stop_all()
print(sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv"))
